# 🚗 YOLOv8 Traffic Detection — Kaggle Training Notebook
**Nhận biết biển giao thông & người đi bộ cho xe tự hành**

| Nhãn | Class | Mô tả |
|------|-------|-------|
| 0 | `stop_sign` | Biển dừng (bát giác đỏ) |
| 1 | `no_entry` | Biển cấm vào |
| 2 | `red_light` | Đèn đỏ |
| 3 | `yellow_light` | Đèn vàng |
| 4 | `green_light` | Đèn xanh |
| 5 | `person` | Người đi bộ |

### Pipeline
`Dataset (Kaggle Hub / upload)` → `Verify` → `Train YOLOv8` → `Validate` → `Export ONNX` → `Download`


## 0. Kiểm tra GPU & Môi trường

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

# Kiểm tra GPU
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU:", result.stdout.strip())
else:
    print("⚠️  Không phát hiện GPU — vào Notebook Settings → Accelerator → GPU T4 x2")

# Python version
print(f"🐍 Python {sys.version.split()[0]}")
print(f"📁 Working dir: {Path.cwd()}")


## 1. Cài đặt thư viện

In [ ]:
%%capture install_log
!pip install ultralytics>=8.2 opencv-python-headless pyyaml onnx onnxsim -q

import ultralytics, cv2, yaml
print(f"✅ ultralytics {ultralytics.__version__}")
print(f"✅ opencv      {cv2.__version__}")


## 2. Cấu hình — chỉnh tại đây

In [ ]:
# ════════════════════════════════════════════════════
#  ⚙️  THAY ĐỔI CÁC THAM SỐ NÀY THEO NHU CẦU
# ════════════════════════════════════════════════════

# Nguồn dataset
# - "sample"  : tạo dataset synthetic để test pipeline (không cần upload gì)
# - "kaggle"  : tải từ Kaggle Dataset (điền KAGGLE_DATASET bên dưới)
# - "upload"  : dataset đã upload lên /kaggle/input/<tên-dataset>
DATASET_SOURCE = "sample"

# Kaggle dataset slug (chỉ dùng khi DATASET_SOURCE = "kaggle")
# Ví dụ: "valentynsichkar/traffic-sign-classifications-in-practice"
KAGGLE_DATASET = "your-username/your-dataset-slug"

# Path dataset đã upload (chỉ dùng khi DATASET_SOURCE = "upload")
# Ví dụ: "/kaggle/input/my-traffic-dataset"
UPLOAD_DATASET_PATH = "/kaggle/input/your-dataset-name"

# ── Model & Training ─────────────────────────────────
MODEL_NAME  = "yolov8n"   # yolov8n (nhanh) | yolov8s | yolov8m (chính xác hơn)
EPOCHS      = 50          # Số epoch — 50 đủ để demo, 100-200 cho kết quả tốt hơn
IMG_SIZE    = 640         # 640 (chuẩn YOLO)
BATCH_SIZE  = 16          # Tăng lên 32 nếu dùng GPU T4 x2 (16GB VRAM)
PATIENCE    = 30          # Early stopping: dừng sau N epoch không cải thiện

# ── Classes ──────────────────────────────────────────
CLASSES = ["stop_sign", "no_entry", "red_light", "yellow_light", "green_light", "person"]
NUM_CLASSES = len(CLASSES)

# ── Paths ─────────────────────────────────────────────
DATASET_DIR  = Path("/kaggle/working/dataset")
RUNS_DIR     = Path("/kaggle/working/runs/train")
OUTPUT_DIR   = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model     : {MODEL_NAME}")
print(f"Epochs    : {EPOCHS}")
print(f"Batch     : {BATCH_SIZE}")
print(f"Classes   : {CLASSES}")
print(f"Dataset   : {DATASET_SOURCE}")


## 3. Chuẩn bị Dataset

In [ ]:
import numpy as np
import cv2
import shutil

def create_sample_dataset(dataset_dir: Path, n_train=120, n_val=30):
    """Tạo dataset synthetic để test pipeline — thay bằng ảnh thật khi deploy."""
    print("📦 Tạo dataset mẫu (synthetic)...")
    print("   ⚠️  Chỉ dùng để kiểm tra pipeline — dùng ảnh thật để train thực tế!")

    OBJ_COLORS = {
        0: (0,0,200), 1: (0,0,180), 2: (0,0,255),
        3: (0,165,255), 4: (0,255,0), 5: (255,200,100),
    }
    np.random.seed(42)

    for split, n_imgs in [("train", n_train), ("val", n_val)]:
        img_dir = dataset_dir / "images" / split
        lbl_dir = dataset_dir / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for idx in range(n_imgs):
            img = np.full((480, 640, 3), [50, 80, 50], dtype=np.uint8)
            # Thêm background texture đơn giản
            for _ in range(20):
                x1, y1 = np.random.randint(0,640), np.random.randint(0,480)
                cv2.line(img, (x1,y1), (x1+np.random.randint(-100,100), y1+np.random.randint(-100,100)),
                         (60,90,60), 1)

            n_objs = np.random.randint(1, 4)
            labels = []
            for _ in range(n_objs):
                cls = np.random.randint(0, NUM_CLASSES)
                w = np.random.randint(40, 120)
                h = np.random.randint(40, 120)
                x = np.random.randint(0, max(1, 640 - w))
                y = np.random.randint(0, max(1, 480 - h))
                cv2.rectangle(img, (x, y), (x+w, y+h), OBJ_COLORS[cls], -1)
                cv2.rectangle(img, (x, y), (x+w, y+h), (255,255,255), 1)
                cv2.putText(img, CLASSES[cls][:5], (x+2, y+h//2),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,255), 1)
                cx = (x + w/2) / 640
                cy = (y + h/2) / 480
                nw = w / 640
                nh = h / 480
                labels.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

            cv2.imwrite(str(img_dir / f"img_{idx:04d}.jpg"), img)
            with open(lbl_dir / f"img_{idx:04d}.txt", "w") as f:
                f.write("\n".join(labels))

    print(f"   ✅ Train: {n_train} ảnh | Val: {n_val} ảnh")


def setup_kaggle_dataset(slug: str, dataset_dir: Path):
    """Tải dataset từ Kaggle Hub về và giả định cấu trúc YOLO."""
    import subprocess
    print(f"⬇️  Tải Kaggle dataset: {slug}")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dataset_dir), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ Lỗi tải dataset:\n{result.stderr}")
        print("   Hãy kiểm tra Kaggle API key và slug dataset")
    else:
        print(f"✅ Đã tải vào: {dataset_dir}")
        print("   Kiểm tra cấu trúc thư mục — đảm bảo có images/train, images/val, labels/train, labels/val")


def setup_upload_dataset(src_path: str, dataset_dir: Path):
    """Copy dataset từ /kaggle/input sang working directory."""
    src = Path(src_path)
    if not src.exists():
        print(f"❌ Không tìm thấy: {src}")
        print("   Hãy thêm dataset vào notebook qua 'Add data' → 'Upload'")
        return False
    print(f"📂 Sao chép dataset từ {src} ...")
    shutil.copytree(str(src), str(dataset_dir), dirs_exist_ok=True)
    print(f"✅ Đã copy sang: {dataset_dir}")
    return True


# ── Thực thi theo DATASET_SOURCE ────────────────────────────────────────────
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

if DATASET_SOURCE == "sample":
    create_sample_dataset(DATASET_DIR)

elif DATASET_SOURCE == "kaggle":
    setup_kaggle_dataset(KAGGLE_DATASET, DATASET_DIR)

elif DATASET_SOURCE == "upload":
    setup_upload_dataset(UPLOAD_DATASET_PATH, DATASET_DIR)

else:
    print(f"❌ DATASET_SOURCE không hợp lệ: '{DATASET_SOURCE}'")
    print("   Chọn một trong: 'sample', 'kaggle', 'upload'")


## 4. Xem trước Dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

COLORS_VIS = [
    "#e74c3c","#c0392b","#e67e22","#f39c12","#2ecc71","#3498db"
]

def draw_yolo_labels(img_path, lbl_path, ax):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = mpatches.Rectangle((x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor=COLORS_VIS[cls % len(COLORS_VIS)],
                    facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, max(0, y1-3), CLASSES[cls],
                        fontsize=7, color=COLORS_VIS[cls % len(COLORS_VIS)],
                        fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.1", facecolor="white", alpha=0.6))

# Lấy tối đa 8 ảnh từ train set
train_imgs = sorted((DATASET_DIR / "images" / "train").glob("*.jpg"))[:8]
if not train_imgs:
    train_imgs = sorted((DATASET_DIR / "images" / "train").glob("*.png"))[:8]

if train_imgs:
    n = min(len(train_imgs), 8)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else list(axes)

    for i, img_path in enumerate(train_imgs):
        lbl_path = DATASET_DIR / "labels" / "train" / (img_path.stem + ".txt")
        draw_yolo_labels(img_path, lbl_path, axes[i])

    for j in range(i+1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Xem trước Train Set (với bounding box)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Không tìm thấy ảnh trong dataset/images/train/")


## 5. Kiểm tra Dataset & Tạo dataset.yaml

In [ ]:
import yaml

def create_dataset_yaml(dataset_dir: Path) -> Path:
    yaml_path = dataset_dir / "dataset.yaml"
    cfg = {
        "path"  : str(dataset_dir.resolve()),
        "train" : "images/train",
        "val"   : "images/val",
        "nc"    : NUM_CLASSES,
        "names" : CLASSES,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    return yaml_path


def verify_dataset(dataset_dir: Path) -> bool:
    print("═" * 55)
    print("  KIỂM TRA DATASET")
    print("═" * 55)
    ok = True
    stats = {}

    for split in ["train", "val"]:
        img_dir = dataset_dir / "images" / split
        lbl_dir = dataset_dir / "labels" / split

        for d in [img_dir, lbl_dir]:
            if not d.exists():
                print(f"  ❌ Thiếu thư mục: {d}")
                ok = False

        if not ok:
            continue

        imgs = (list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
                + list(img_dir.glob("*.jpeg")))
        lbls = list(lbl_dir.glob("*.txt"))

        counts = [0] * NUM_CLASSES
        errors = 0
        for lf in lbls:
            try:
                for line in lf.read_text().splitlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cid = int(parts[0])
                        if 0 <= cid < NUM_CLASSES:
                            counts[cid] += 1
                        else:
                            errors += 1
                    elif line.strip():
                        errors += 1
            except Exception as e:
                errors += 1

        stats[split] = {"imgs": len(imgs), "lbls": len(lbls), "counts": counts, "errors": errors}

        unmatched = {p.stem for p in imgs} - {p.stem for p in lbls}
        print(f"\n  [{split.upper()}]  ảnh: {len(imgs):4d}   nhãn: {len(lbls):4d}")
        if unmatched:
            print(f"   ⚠️  {len(unmatched)} ảnh không có nhãn tương ứng")
        if errors:
            print(f"   ⚠️  {errors} dòng nhãn bị lỗi format")
        if sum(counts) == 0 and len(imgs) > 0:
            ok = False

        print(f"\n   Phân bổ nhãn:")
        for i, (cls_name, cnt) in enumerate(zip(CLASSES, counts)):
            bar = "█" * min(cnt // max(sum(counts)//40, 1), 30)
            print(f"   [{i}] {cls_name:<15} {cnt:5d}  {bar}")

    print()
    if ok:
        print("  ✅ Dataset hợp lệ — sẵn sàng train!")
    else:
        print("  ❌ Dataset có vấn đề — hãy kiểm tra lại")
    print("═" * 55)
    return ok, stats


yaml_path = create_dataset_yaml(DATASET_DIR)
print(f"✅ dataset.yaml: {yaml_path}\n")
is_valid, dataset_stats = verify_dataset(DATASET_DIR)


In [ ]:
# Biểu đồ phân bổ nhãn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, split in zip(axes, ["train", "val"]):
    if split not in dataset_stats:
        continue
    counts = dataset_stats[split]["counts"]
    bars = ax.bar(CLASSES, counts, color=COLORS_VIS[:NUM_CLASSES], edgecolor="white", linewidth=0.8)
    ax.set_title(f"Phân bổ nhãn — {split.upper()}", fontweight="bold")
    ax.set_ylabel("Số lượng annotation")
    ax.set_xlabel("Class")
    ax.tick_params(axis="x", rotation=25)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(cnt), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


## 6. Train YOLOv8

In [ ]:
from ultralytics import YOLO
import torch

# Tự chọn device
device = "0" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️  GPU: {gpu_name} ({vram:.1f} GB VRAM)")
    # Tự động tăng batch nếu VRAM đủ lớn
    if vram >= 14:
        BATCH_SIZE = 32
        print(f"   ↑ Tự động tăng batch → {BATCH_SIZE} (VRAM ≥ 14GB)")
else:
    print("⚠️  Chạy trên CPU — sẽ rất chậm. Hãy bật GPU!")

print(f"\n🚀 Bắt đầu train: {MODEL_NAME}  |  {EPOCHS} epochs  |  batch={BATCH_SIZE}")
print(f"   Dataset: {yaml_path}")
print()

# Load pretrained model
model = YOLO(f"{MODEL_NAME}.pt")

# Train
results = model.train(
    data         = str(yaml_path),
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH_SIZE,
    project      = str(RUNS_DIR),
    name         = "traffic_detector",
    exist_ok     = True,
    device       = device,

    # ── Augmentation (tối ưu cho biển nhỏ & đèn) ─────────────────────
    hsv_h        = 0.015,
    hsv_s        = 0.5,
    hsv_v        = 0.4,
    degrees      = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    shear        = 2.0,
    perspective  = 0.0005,
    flipud       = 0.0,      # KHÔNG lật dọc — biển không bao giờ ngược
    fliplr       = 0.5,
    mosaic       = 1.0,      # Mosaic rất hiệu quả cho small object
    mixup        = 0.1,
    copy_paste   = 0.1,

    # ── Optimizer ────────────────────────────────────────────────────
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs   = 3,
    warmup_momentum = 0.8,

    # ── Loss weights ─────────────────────────────────────────────────
    box = 7.5,
    cls = 0.5,
    dfl = 1.5,

    # ── Misc ─────────────────────────────────────────────────────────
    patience     = PATIENCE,
    save_period  = 10,
    plots        = True,
    verbose      = True,
)

print("\n✅ Training hoàn tất!")


## 7. Validation & Kết quả

In [ ]:
best_pt = RUNS_DIR / "traffic_detector" / "weights" / "best.pt"

if not best_pt.exists():
    candidates = sorted(RUNS_DIR.rglob("best.pt"), key=lambda p: p.stat().st_mtime)
    best_pt = candidates[-1] if candidates else None

if best_pt and best_pt.exists():
    print(f"📍 Best weights: {best_pt}\n")
    val_model = YOLO(str(best_pt))
    metrics   = val_model.val(data=str(yaml_path), imgsz=IMG_SIZE, device=device, verbose=False)

    print("┌─────────────────────────────────────────┐")
    print("│          KẾT QUẢ VALIDATION             │")
    print("├─────────────────────────────────────────┤")
    print(f"│  mAP@0.5       : {metrics.box.map50:.4f}                │")
    print(f"│  mAP@0.5:0.95  : {metrics.box.map:.4f}                │")
    print(f"│  Precision     : {metrics.box.mp:.4f}                │")
    print(f"│  Recall        : {metrics.box.mr:.4f}                │")
    print("└─────────────────────────────────────────┘")

    # Per-class metrics
    print("\n  Metrics theo class:")
    for i, cls_name in enumerate(CLASSES):
        try:
            ap50 = metrics.box.ap50[i]
            print(f"   [{i}] {cls_name:<15} AP@50={ap50:.4f}")
        except (IndexError, AttributeError):
            pass
else:
    print("❌ Không tìm thấy best.pt — kiểm tra lại bước train")


In [ ]:
import pandas as pd

results_csv = RUNS_DIR / "traffic_detector" / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Training Curves", fontsize=14, fontweight="bold")

    plot_pairs = [
        ("train/box_loss", "val/box_loss", "Box Loss",   axes[0][0]),
        ("train/cls_loss", "val/cls_loss", "Class Loss", axes[0][1]),
        ("train/dfl_loss", "val/dfl_loss", "DFL Loss",   axes[0][2]),
        ("metrics/precision(B)", None, "Precision",      axes[1][0]),
        ("metrics/recall(B)",    None, "Recall",         axes[1][1]),
        ("metrics/mAP50(B)",     None, "mAP@0.5",        axes[1][2]),
    ]

    for train_col, val_col, title, ax in plot_pairs:
        if train_col in df.columns:
            ax.plot(df[train_col], label="train", color="#3498db", linewidth=1.8)
        if val_col and val_col in df.columns:
            ax.plot(df[val_col], label="val", color="#e74c3c", linewidth=1.8, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / "training_curves.png"), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ Đã lưu: {OUTPUT_DIR / 'training_curves.png'}")
else:
    print(f"⚠️  Không tìm thấy results.csv tại {results_csv}")


In [ ]:
# Hiển thị confusion matrix và các biểu đồ được tạo bởi YOLO
run_dir = RUNS_DIR / "traffic_detector"

plot_files = {
    "Confusion Matrix": run_dir / "confusion_matrix_normalized.png",
    "PR Curve"        : run_dir / "PR_curve.png",
    "F1 Curve"        : run_dir / "F1_curve.png",
}

available = {k: v for k, v in plot_files.items() if v.exists()}

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(7*len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, (title, fpath) in zip(axes, available.items()):
        img = plt.imread(str(fpath))
        ax.imshow(img)
        ax.set_title(title, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Chưa tìm thấy biểu đồ — hãy chạy sau khi train xong")


## 8. Inference trên Val Set

In [ ]:
if best_pt and best_pt.exists():
    infer_model = YOLO(str(best_pt))

    val_imgs = (list((DATASET_DIR / "images" / "val").glob("*.jpg"))
              + list((DATASET_DIR / "images" / "val").glob("*.png")))[:6]

    if val_imgs:
        n = len(val_imgs)
        cols = min(3, n)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
        axes = np.array(axes).flatten()

        for i, img_path in enumerate(val_imgs):
            result  = infer_model.predict(str(img_path), imgsz=IMG_SIZE,
                                          conf=0.25, verbose=False)[0]
            plotted = result.plot()  # BGR numpy array
            plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
            axes[i].imshow(plotted)
            axes[i].set_title(img_path.name, fontsize=9)
            axes[i].axis("off")

        for j in range(i+1, len(axes)):
            axes[j].axis("off")

        plt.suptitle("Kết quả Inference trên Val Set", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / "inference_preview.png"), dpi=100, bbox_inches="tight")
        plt.show()
        print(f"✅ Đã lưu: {OUTPUT_DIR / 'inference_preview.png'}")
    else:
        print("⚠️  Không có ảnh val để inference")


## 9. Export ONNX (cho OpenCV / Raspberry Pi)

In [ ]:
if best_pt and best_pt.exists():
    print(f"📤 Exporting {best_pt.name} → ONNX ...")
    export_model = YOLO(str(best_pt))
    onnx_result  = export_model.export(
        format   = "onnx",
        imgsz    = IMG_SIZE,
        opset    = 12,       # opset 12 tương thích tốt nhất OpenCV 4.x
        simplify = True,     # onnx-simplifier: giảm node dư thừa
        dynamic  = False,    # fixed batch=1 cho real-time inference
        half     = False,    # FP32 để tương thích RPi (không GPU)
    )

    onnx_path = Path(str(onnx_result))
    if onnx_path.exists():
        # Copy sang output
        out_onnx = OUTPUT_DIR / "traffic_detector_best.onnx"
        import shutil
        shutil.copy2(str(onnx_path), str(out_onnx))
        print(f"\n✅ ONNX đã lưu: {out_onnx}")
        print(f"   Size: {out_onnx.stat().st_size / 1e6:.2f} MB")
        print(f"\n   Dùng trong xe tự hành:")
        print(f"   python3 cv_lane_driver_v2.py --yolo-model traffic_detector_best.onnx")
else:
    print("⚠️  Chưa có best.pt để export")


## 10. Đóng gói & Tải về

In [ ]:
import zipfile, shutil

zip_path = Path("/kaggle/working/traffic_detector_results.zip")

with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
    # Best weights .pt
    if best_pt and best_pt.exists():
        zf.write(str(best_pt), "weights/best.pt")

    # ONNX
    onnx_out = OUTPUT_DIR / "traffic_detector_best.onnx"
    if onnx_out.exists():
        zf.write(str(onnx_out), "weights/traffic_detector_best.onnx")

    # Plots
    for f in OUTPUT_DIR.glob("*.png"):
        zf.write(str(f), f"plots/{f.name}")
    for f in (RUNS_DIR / "traffic_detector").glob("*.png"):
        zf.write(str(f), f"plots/{f.name}")

    # dataset.yaml (để tái sử dụng)
    zf.write(str(yaml_path), "dataset.yaml")

    # results.csv
    if results_csv.exists():
        zf.write(str(results_csv), "results.csv")

size_mb = zip_path.stat().st_size / 1e6
print(f"📦 Đã tạo: {zip_path}")
print(f"   Kích thước: {size_mb:.1f} MB")
print()
print("📥 Cách tải về:")
print("   Kaggle > Notebook > Output > traffic_detector_results.zip > Download")
print()
print("Nội dung zip:")
with zipfile.ZipFile(str(zip_path), "r") as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"   {name:<50} {info.file_size/1e3:8.1f} KB")


## 11. Hướng dẫn sử dụng tiếp theo

### Dùng ONNX với OpenCV trên xe tự hành

```python
import cv2
import numpy as np

net = cv2.dnn.readNetFromONNX("traffic_detector_best.onnx")
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

CLASSES = ["stop_sign", "no_entry", "red_light", "yellow_light", "green_light", "person"]
CONF_THRESH = 0.5
NMS_THRESH  = 0.4

def detect(frame):
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (640, 640), swapRB=True)
    net.setInput(blob)
    outputs = net.forward()
    # ... parse outputs → boxes, confs, class_ids
    return detections
```

### Tiếp tục train với dataset thật
1. **Thu thập ảnh** từ camera xe hoặc tải từ Kaggle/Roboflow
2. **Annotate** bằng [LabelImg](https://github.com/HumanSignal/labelImg) hoặc [Roboflow](https://roboflow.com) (YOLO format)
3. Đặt `DATASET_SOURCE = "upload"` và upload lên Kaggle
4. Tăng `EPOCHS = 100` và `MODEL_NAME = "yolov8s"` để kết quả tốt hơn

### Dataset gợi ý trên Kaggle
- `valentynsichkar/traffic-sign-classifications-in-practice`
- `andrewmvd/road-sign-detection`
- `pkdarabi/cardetection` (có person + vehicle)
